# 🔐 Vuln Detector v2 — GraphCodeBERT **Full Fine-tune** + **Multi-task MIL**
Cải tiến từ bản v1 (AUC 0.80 / Acc 0.73). Đầu vào: **`rag_knowledge.jsonl`** (có thêm `categories`/`swc_ids`).

**Vì sao v1 điểm chưa cao & v2 sửa gì:**

| Nguyên nhân v1 | Cách v2 xử lý |
|---|---|
| CodeBERT **không** pretrain trên Solidity + LoRA chỉ 0.7% params → underfit | **GraphCodeBERT + FULL fine-tune** (mở khoá toàn bộ encoder). LR hạ về **2e-5** cho full-FT |
| Nhãn nhị phân mờ, model học tín hiệu "trông có vẻ vuln" | **Multi-task**: học song song nhãn nhị phân + **loại lỗ hổng** (reentrancy, arithmetic…) → buộc học pattern cụ thể |
| Attention pool dễ "trung bình hoá" các chunk | **Attention-entropy regularization** → dồn chú ý vào chunk chứa lỗi |
| Không biết nguồn nào kéo điểm | **Báo cáo per-source** (smartbug vs dappscan vs solodit) + in đường cong train/val |

> Kaggle: **Add Data → upload `rag_knowledge.jsonl`**. Bật GPU. Chạy cell cài đặt → **Restart kernel** → Run All.

## ⚙️ 0. Cài đặt (ghim bản 4.x ổn định) — chạy rồi **RESTART KERNEL**

In [1]:
# Ghim bộ thư viện tương thích (tránh transformers 5.x lỗi trên Kaggle).
!pip install -q "transformers==4.46.3" "tokenizers==0.20.3" "huggingface_hub==0.25.2" "peft==0.13.2" "accelerate==1.0.1" "datasets==3.0.1" scikit-learn sentencepiece
import importlib
for pkg in ["transformers", "tokenizers", "huggingface_hub", "accelerate", "datasets", "torch"]:
    try:
        print(f"  {pkg}: {getattr(importlib.import_module(pkg), '__version__', '?')}")
    except Exception as e:
        print(f"  {pkg}: !! {e}")
print("\n🔴 BẮT BUỘC: Run -> Restart Kernel, rồi Run All. (transformers 5.x đã nạp sẵn -> phải restart.)")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 70.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 79.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 436.6/436.6 kB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 330.9/330.9 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 471.6/471.6 kB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 6.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
s3fs 2026

## 📦 1. Cấu hình

In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"]  = "0"   # collator gộp-chunk-theo-contract KHÔNG hợp DataParallel -> 1 GPU
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import json, re, glob, hashlib, random, collections
import numpy as np
import torch, torch.nn as nn
import torch.nn.functional as F

import transformers
assert int(transformers.__version__.split(".")[0]) < 5, \
    f"transformers=={transformers.__version__}: CHƯA restart kernel! Restart rồi Run All."
print("transformers:", transformers.__version__)

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

# ----------------------- HYPER-PARAMS -----------------------
MODEL_NAME     = "microsoft/graphcodebert-base"   # mạnh hơn codebert cho code
TOKENIZER_NAME = "roberta-base"                    # có tokenizer.json, cùng vocab -> nạp thẳng
MAX_LEN    = 512
MAX_CHUNKS = 8
LABEL2ID   = {"Safe": 0, "Vulnerable": 1}
ID2LABEL   = {0: "Safe", 1: "Vulnerable"}

# ── Capacity: FULL fine-tune (không LoRA) ──
USE_LORA        = False        # False = full fine-tune toàn bộ encoder (theo lựa chọn)
LORA_R, LORA_ALPHA, LORA_DROPOUT = 32, 64, 0.05   # chỉ dùng nếu USE_LORA=True
GRAD_CHECKPOINT = True

# ── Multi-task + regularization ──
USE_MULTITASK   = True         # học thêm loại lỗ hổng (cần rag_knowledge.jsonl)
AUX_WEIGHT      = 0.3          # trọng số nhánh phân loại loại lỗ hổng
ENT_REG         = 0.01         # phạt entropy attention -> dồn chú ý vào chunk chứa lỗi
LABEL_SMOOTHING = 0.05
USE_CLASS_WEIGHT= True

# ── Train ── (full-FT dùng LR NHỎ; OOM -> giảm TRAIN_BS 4->2 hoặc MAX_CHUNKS 8->6)
EPOCHS     = 10
TRAIN_BS   = 4
EVAL_BS    = 8
GRAD_ACCUM = 4
LR         = 2e-5 if not USE_LORA else 2e-4   # full-FT cần LR nhỏ hơn LoRA ~10x
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06
EARLY_STOP_PATIENCE = 3

OUTPUT_DIR = "./vuln-v2"
SAVE_DIR   = "./vuln-v2-final"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print(f"Model: {MODEL_NAME} | full_finetune={not USE_LORA} | LR={LR} | multitask={USE_MULTITASK}")

transformers: 4.46.3
Device: Tesla T4
Model: microsoft/graphcodebert-base | full_finetune=True | LR=2e-05 | multitask=True


## 📂 2. Nạp dữ liệu (`rag_knowledge.jsonl`) + xây taxonomy loại lỗ hổng

In [3]:
def find_data_path():
    for pat in ["/kaggle/input/**/rag_knowledge.jsonl", "rag_knowledge.jsonl",
                "data/clean_dataset/rag_knowledge.jsonl"]:
        hits = sorted(glob.glob(pat, recursive=True))
        if hits: return hits[0], True
    # fallback: detect_dataset (không có categories -> multitask tắt)
    for pat in ["/kaggle/input/**/detect_dataset.jsonl", "detect_dataset.jsonl",
                "data/clean_dataset/detect_dataset.jsonl"]:
        hits = sorted(glob.glob(pat, recursive=True))
        if hits: return hits[0], False
    raise FileNotFoundError("Hãy upload rag_knowledge.jsonl (hoặc detect_dataset.jsonl) lên Kaggle.")

DATA_PATH, HAS_CATS = find_data_path()
print("DATA_PATH =", DATA_PATH, "| có categories:", HAS_CATS)
if USE_MULTITASK and not HAS_CATS:
    print("⚠️ Không có categories -> tắt multi-task."); USE_MULTITASK = False

rows = [json.loads(l) for l in open(DATA_PATH, encoding="utf-8") if l.strip()]
rows = [r for r in rows if r.get("code") and r.get("label") in LABEL2ID]
for r in rows:
    r.setdefault("categories", []); r.setdefault("source", "unknown")

# Taxonomy loại lỗ hổng (union các categories xuất hiện)
CATS = sorted({c for r in rows for c in (r.get("categories") or [])})
if not CATS: CATS = ["_none_"]
CAT2I = {c: i for i, c in enumerate(CATS)}
n, nv = len(rows), sum(r["label"] == "Vulnerable" for r in rows)
# aux_valid: Safe (target toàn 0 hợp lệ) hoặc Vulnerable có categories
n_auxvalid = sum(1 for r in rows if r["label"] == "Safe" or len(r.get("categories") or []) > 0)
print(f"Mẫu: {n:,} | Vuln {nv:,} ({100*nv/n:.1f}%) | Safe {n-nv:,}")
print(f"Loại lỗ hổng ({len(CATS)}): {CATS}")
print(f"Số mẫu có nhãn-loại hợp lệ cho multi-task: {n_auxvalid:,}/{n:,}")
print("Theo nguồn:", dict(collections.Counter(r["source"] for r in rows)))

DATA_PATH = /kaggle/input/datasets/thanhphuocjr/rag-knowledge/rag_knowledge.jsonl | có categories: True
Mẫu: 8,218 | Vuln 4,235 (51.5%) | Safe 3,983
Loại lỗ hổng (9): ['access_control', 'arithmetic', 'bad_randomness', 'denial_service', 'front_running', 'other', 'reentrancy', 'time_manipulation', 'unchecked_low_calls']
Số mẫu có nhãn-loại hợp lệ cho multi-task: 7,166/8,218
Theo nguồn: {'smartbug_wild': 5132, 'dappscan': 575, 'solodit': 2511}


## ✂️ 3. Chia tập chống rò rỉ (group theo chữ ký cấu trúc)

In [4]:
def structural_sig(code):
    s = re.sub(r"\b[A-Za-z_]\w*\b", "X", code)
    return hashlib.md5(re.sub(r"\s+", "", s).encode("utf-8", "replace")).hexdigest()

groups = collections.defaultdict(list)
for i, r in enumerate(rows):
    groups[structural_sig(r["code"])].append(i)
gkeys = list(groups.keys()); random.Random(SEED).shuffle(gkeys)

nt = int(0.10 * n); nvl = int(0.10 * n)
test_idx, val_idx, train_idx = [], [], []
for k in gkeys:
    g = groups[k]
    if len(test_idx) + len(g) <= nt:   test_idx += g
    elif len(val_idx) + len(g) <= nvl: val_idx += g
    else:                              train_idx += g
take = lambda ix: [rows[i] for i in ix]
train_raw, val_raw, test_raw = take(train_idx), take(val_idx), take(test_idx)

def bal(nm, d):
    v = sum(x["label"] == "Vulnerable" for x in d)
    print(f"  {nm:5s}: {len(d):5,d} | Vuln {v:5,d} ({100*v/max(1,len(d)):.1f}%)  Safe {len(d)-v:5,d}")
print(f"Nhóm cấu trúc: {len(gkeys):,}"); bal("train", train_raw); bal("val", val_raw); bal("test", test_raw)

Nhóm cấu trúc: 7,766
  train: 6,576 | Vuln 3,379 (51.4%)  Safe 3,197
  val  :   821 | Vuln   431 (52.5%)  Safe   390
  test :   821 | Vuln   425 (51.8%)  Safe   396


## 🪟 4. Tokenizer & Chunk hoá (kèm nhãn-loại đa nhãn)

In [5]:
import subprocess, sys
for _p in ["sentencepiece", "tiktoken"]:
    try: importlib.import_module(_p)
    except Exception: subprocess.run([sys.executable, "-m", "pip", "install", "-q", _p], check=False)

from transformers import AutoTokenizer
def load_tokenizer(name):
    try: return AutoTokenizer.from_pretrained(name, use_fast=True)
    except Exception as e:
        print("Fast lỗi -> slow:", str(e)[:100]); return AutoTokenizer.from_pretrained(name, use_fast=False)
tokenizer = load_tokenizer(TOKENIZER_NAME)
for attr, tk in [("bos_token","<s>"),("eos_token","</s>"),("unk_token","<unk>"),
                 ("pad_token","<pad>"),("cls_token","<s>"),("sep_token","</s>"),("mask_token","<mask>")]:
    if getattr(tokenizer, attr, None) is None:
        try: setattr(tokenizer, attr, tk)
        except Exception: pass
CLS = tokenizer.convert_tokens_to_ids("<s>"); SEP = tokenizer.convert_tokens_to_ids("</s>")
PAD = tokenizer.convert_tokens_to_ids("<pad>"); CONTENT = MAX_LEN - 2
assert all(isinstance(x, int) and x >= 0 for x in (CLS, SEP, PAD))
print(f"Tokenizer OK | vocab={tokenizer.vocab_size:,} | CLS/SEP/PAD={CLS}/{SEP}/{PAD}")

def chunk_encode(code):
    ids = tokenizer.encode(code, add_special_tokens=False) or [PAD]
    wins = [ids[i:i+CONTENT] for i in range(0, len(ids), CONTENT)]
    if len(wins) > MAX_CHUNKS:
        sel = sorted(set(np.linspace(0, len(wins)-1, MAX_CHUNKS).round().astype(int).tolist()))
        wins = [wins[i] for i in sel]
    iid, am = [], []
    for w in wins:
        seq = [CLS] + w + [SEP]; m = [1]*len(seq)
        if len(seq) < MAX_LEN:
            p = MAX_LEN - len(seq); seq += [PAD]*p; m += [0]*p
        iid.append(seq); am.append(m)
    return torch.tensor(iid, dtype=torch.long), torch.tensor(am, dtype=torch.long)

def cat_vector(item):
    v = torch.zeros(len(CATS), dtype=torch.float)
    for c in (item.get("categories") or []):
        if c in CAT2I: v[CAT2I[c]] = 1.0
    valid = 1.0 if (item["label"] == "Safe" or len(item.get("categories") or []) > 0) else 0.0
    return v, valid

class ChunkedDS(torch.utils.data.Dataset):
    def __init__(self, items):
        self.data = []
        for it in items:
            iid, am = chunk_encode(it["code"])
            cv, valid = cat_vector(it)
            self.data.append({"input_ids": iid, "attention_mask": am,
                              "label": LABEL2ID[it["label"]], "cat_target": cv, "aux_valid": valid})
    def __len__(self): return len(self.data)
    def __getitem__(self, i): return self.data[i]

print("Đang chunk hoá...")
train_ds, val_ds, test_ds = ChunkedDS(train_raw), ChunkedDS(val_raw), ChunkedDS(test_raw)
tc = sum(d["input_ids"].shape[0] for d in train_ds.data)
print(f"Train: {len(train_ds):,} contract -> {tc:,} chunk (TB {tc/len(train_ds):.2f}/contract)")

def collate(fs):
    return {"input_ids":      torch.cat([f["input_ids"] for f in fs], 0),
            "attention_mask": torch.cat([f["attention_mask"] for f in fs], 0),
            "n_chunks":       torch.tensor([f["input_ids"].shape[0] for f in fs], dtype=torch.long),
            "labels":         torch.tensor([f["label"] for f in fs], dtype=torch.long),
            "cat_target":     torch.stack([f["cat_target"] for f in fs], 0),
            "aux_valid":      torch.tensor([f["aux_valid"] for f in fs], dtype=torch.float)}

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (529 > 512). Running this sequence through the model will result in indexing errors


Tokenizer OK | vocab=50,265 | CLS/SEP/PAD=0/2/1
Đang chunk hoá...
Train: 6,576 contract -> 27,180 chunk (TB 4.13/contract)


## 🤖 5. Mô hình: Full fine-tune + MIL attention + 2 nhánh (nhị phân & loại lỗ hổng)

In [6]:
from transformers import AutoModel
from transformers.modeling_outputs import SequenceClassifierOutput
from peft import LoraConfig, get_peft_model, TaskType

class HierMultiTask(nn.Module):
    def __init__(self, encoder, hidden, num_cats, dropout=0.1, class_weights=None,
                 label_smoothing=0.0, aux_weight=0.0, ent_reg=0.0):
        super().__init__()
        self.encoder = encoder
        self.attn_V = nn.Linear(hidden, 128); self.attn_U = nn.Linear(hidden, 128)
        self.attn_w = nn.Linear(128, 1)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden, 2)          # nhánh nhị phân
        self.cat_head   = nn.Linear(hidden, num_cats)    # nhánh loại lỗ hổng
        self.label_smoothing = label_smoothing
        self.aux_weight = aux_weight; self.ent_reg = ent_reg
        if class_weights is not None: self.register_buffer("class_weights", class_weights)
        else: self.class_weights = None

    def forward(self, input_ids, attention_mask, n_chunks, labels=None, cat_target=None, aux_valid=None):
        h = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state[:, 0]
        A = self.attn_w(torch.tanh(self.attn_V(h)) * torch.sigmoid(self.attn_U(h)))   # [N,1]
        pooled, ent, idx = [], h.new_zeros(()), 0
        for c in n_chunks:
            c = int(c); ai = torch.softmax(A[idx:idx+c], dim=0)
            pooled.append((ai * h[idx:idx+c]).sum(0))
            if c > 1:
                p = ai.squeeze(-1).clamp_min(1e-9); ent = ent + -(p * torch.log(p)).sum()
            idx += c
        z = self.dropout(torch.stack(pooled, 0))          # [B,H]
        bin_logits = self.classifier(z); cat_logits = self.cat_head(z)
        loss = None
        if labels is not None:
            w = self.class_weights if self.class_weights is not None else None
            loss = F.cross_entropy(bin_logits, labels, weight=w, label_smoothing=self.label_smoothing)
            if cat_target is not None and self.aux_weight > 0:
                bce = F.binary_cross_entropy_with_logits(cat_logits, cat_target, reduction="none").mean(1)
                if aux_valid is not None:
                    loss = loss + self.aux_weight * (bce * aux_valid).sum() / aux_valid.sum().clamp(min=1.0)
                else:
                    loss = loss + self.aux_weight * bce.mean()
            if self.ent_reg > 0:
                loss = loss + self.ent_reg * (ent / max(1, len(n_chunks)))
        return SequenceClassifierOutput(loss=loss, logits=bin_logits)

encoder = AutoModel.from_pretrained(MODEL_NAME, trust_remote_code=True)
HIDDEN = encoder.config.hidden_size
if USE_LORA:
    encoder = get_peft_model(encoder, LoraConfig(
        r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT, bias="none",
        task_type=TaskType.FEATURE_EXTRACTION,
        target_modules=["query","key","value","output.dense","intermediate.dense"]))
    encoder.print_trainable_parameters()
if GRAD_CHECKPOINT:
    try:
        encoder.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        if hasattr(encoder, "enable_input_require_grads"): encoder.enable_input_require_grads()
        print("Gradient checkpointing: ON")
    except Exception as e:
        print("Grad ckpt OFF:", e)

cnt = collections.Counter(d["label"] for d in train_ds.data); N = sum(cnt.values())
cw = torch.tensor([N/(2*cnt[0]), N/(2*cnt[1])], dtype=torch.float) if USE_CLASS_WEIGHT else None
model = HierMultiTask(encoder, HIDDEN, num_cats=len(CATS), dropout=0.1, class_weights=cw,
                      label_smoothing=LABEL_SMOOTHING,
                      aux_weight=(AUX_WEIGHT if USE_MULTITASK else 0.0), ent_reg=ENT_REG).to(device)
print(f"✅ Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,} "
      f"| full_FT={not USE_LORA} | aux_weight={model.aux_weight} | class_weights={cw}")

2026-07-01 06:32:04.663561: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1782887525.120598      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1782887525.270160      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1782887526.440758      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782887526.440803      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782887526.440806      23 computation_placer.cc:177] computation placer alr

config.json:   0%|          | 0.00/539 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at microsoft/graphcodebert-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Gradient checkpointing: ON
✅ Trainable params: 124,851,084 | full_FT=True | aux_weight=0.3 | class_weights=tensor([1.0285, 0.9731])


## 🏋️ 6. Huấn luyện

In [7]:
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
from sklearn.metrics import f1_score

def _lg(p): return p.predictions[0] if isinstance(p.predictions, tuple) else p.predictions
def compute_metrics(p):
    pr = np.argmax(_lg(p), axis=-1)
    return {"f1_macro": f1_score(p.label_ids, pr, average="macro"),
            "f1_vuln":  f1_score(p.label_ids, pr, pos_label=1, average="binary", zero_division=0)}

args = TrainingArguments(
    output_dir=OUTPUT_DIR, num_train_epochs=EPOCHS,
    per_device_train_batch_size=TRAIN_BS, per_device_eval_batch_size=EVAL_BS,
    gradient_accumulation_steps=GRAD_ACCUM, learning_rate=LR, weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO, lr_scheduler_type="cosine", fp16=torch.cuda.is_available(),
    eval_strategy="epoch", save_strategy="epoch", load_best_model_at_end=True,
    metric_for_best_model="f1_macro", greater_is_better=True, logging_steps=50,
    save_total_limit=2, report_to="none", seed=SEED, remove_unused_columns=False,
    dataloader_pin_memory=False, label_names=["labels"], max_grad_norm=1.0,
    save_safetensors=False)   # model là nn.Module tuỳ chỉnh -> dùng torch.save cho an toàn

trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                  data_collator=collate, compute_metrics=compute_metrics,
                  callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOP_PATIENCE)])
print("🚀 Train..."); trainer.train(); print("✅ Xong.")

# --- Đường cong train/val ---
print("\nEpoch |  train_loss |  eval_loss | f1_macro | f1_vuln")
tr = {}
for h in trainer.state.log_history:
    if "loss" in h and "epoch" in h and "eval_loss" not in h:
        tr[round(h["epoch"])] = h["loss"]
for h in trainer.state.log_history:
    if "eval_f1_macro" in h:
        e = round(h["epoch"])
        print(f"  {e:3d} | {tr.get(e, float('nan')):10.4f} | {h['eval_loss']:9.4f} | "
              f"{h['eval_f1_macro']:.4f} | {h['eval_f1_vuln']:.4f}")

🚀 Train...


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Epoch,Training Loss,Validation Loss,F1 Macro,F1 Vuln
1,0.461600,0.440284,0.884243,0.881988
2,0.315100,0.312088,0.940175,0.943089
3,0.235300,0.303192,0.953679,0.954976
4,0.209700,0.253965,0.965853,0.967059
5,0.191600,0.256383,0.963425,0.964539
6,0.163100,0.274072,0.958569,0.959427
7,0.167200,0.239444,0.971957,0.972845
8,0.154200,0.237598,0.969514,0.970554
9,0.157200,0.234591,0.970736,0.971698
10,0.162700,0.239050,0.965868,0.966825


✅ Xong.

Epoch |  train_loss |  eval_loss | f1_macro | f1_vuln
    1 |     0.3920 |    0.4403 | 0.8842 | 0.8820
    2 |     0.3009 |    0.3121 | 0.9402 | 0.9431
    3 |     0.2204 |    0.3032 | 0.9537 | 0.9550
    4 |     0.1972 |    0.2540 | 0.9659 | 0.9671
    5 |     0.1774 |    0.2564 | 0.9634 | 0.9645
    6 |     0.1759 |    0.2741 | 0.9586 | 0.9594
    7 |     0.1671 |    0.2394 | 0.9720 | 0.9728
    8 |     0.1558 |    0.2376 | 0.9695 | 0.9706
    9 |     0.1597 |    0.2346 | 0.9707 | 0.9717
   10 |     0.1627 |    0.2391 | 0.9659 | 0.9668


## 📊 7. Hiệu chỉnh ngưỡng + Đánh giá + **Báo cáo per-source**

In [8]:
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, average_precision_score, f1_score, accuracy_score)

def get_probs(ds):
    out = trainer.predict(ds)
    lg = out.predictions[0] if isinstance(out.predictions, tuple) else out.predictions
    return torch.softmax(torch.tensor(lg), dim=-1)[:, 1].numpy(), out.label_ids

val_probs, val_labels = get_probs(val_ds)
best_t, best_f1 = 0.5, -1.0
for t in np.linspace(0.05, 0.95, 91):
    f1m = f1_score(val_labels, (val_probs >= t).astype(int), average="macro")
    if f1m > best_f1: best_f1, best_t = f1m, float(t)
print(f"Ngưỡng tối ưu (VAL) = {best_t:.3f} | F1-macro(val) = {best_f1:.4f}")

test_probs, test_labels = get_probs(test_ds)
for nm, t in [("0.50", 0.5), (f"{best_t:.3f} (hiệu chỉnh)", best_t)]:
    preds = (test_probs >= t).astype(int)
    print("\n" + "="*60 + f"\n📊 TEST @ threshold {nm}\n" + "="*60)
    print(classification_report(test_labels, preds, target_names=["Safe","Vulnerable"], digits=4, zero_division=0))
    print("Confusion [[TN FP][FN TP]]:\n", confusion_matrix(test_labels, preds))
print(f"\nROC-AUC = {roc_auc_score(test_labels, test_probs):.4f} | PR-AUC = {average_precision_score(test_labels, test_probs):.4f}")

# --- Per-source: nguồn nào kéo điểm? ---
preds = (test_probs >= best_t).astype(int)
src = [r.get("source", "unknown") for r in test_raw]
print("\n--- Điểm theo NGUỒN (test) ---")
print(f"{'source':16s} {'n':>5s} {'acc':>7s} {'f1_macro':>9s} {'AUC':>7s}")
for s in sorted(set(src)):
    ix = [i for i, x in enumerate(src) if x == s]
    yt = test_labels[ix]; yp = preds[ix]; pp = test_probs[ix]
    auc = roc_auc_score(yt, pp) if len(set(yt)) > 1 else float("nan")
    print(f"{s:16s} {len(ix):5d} {accuracy_score(yt, yp):7.3f} "
          f"{f1_score(yt, yp, average='macro'):9.3f} {auc:7.3f}")

Ngưỡng tối ưu (VAL) = 0.160 | F1-macro(val) = 0.9744



📊 TEST @ threshold 0.50
              precision    recall  f1-score   support

        Safe     0.9425    0.9520    0.9472       396
  Vulnerable     0.9549    0.9459    0.9504       425

    accuracy                         0.9488       821
   macro avg     0.9487    0.9490    0.9488       821
weighted avg     0.9489    0.9488    0.9489       821

Confusion [[TN FP][FN TP]]:
 [[377  19]
 [ 23 402]]

📊 TEST @ threshold 0.160 (hiệu chỉnh)
              precision    recall  f1-score   support

        Safe     0.9563    0.9394    0.9478       396
  Vulnerable     0.9444    0.9600    0.9522       425

    accuracy                         0.9501       821
   macro avg     0.9504    0.9497    0.9500       821
weighted avg     0.9502    0.9501    0.9500       821

Confusion [[TN FP][FN TP]]:
 [[372  24]
 [ 17 408]]

ROC-AUC = 0.9717 | PR-AUC = 0.9617

--- Điểm theo NGUỒN (test) ---
source               n     acc  f1_macro     AUC
dappscan            55   0.964     0.491     nan
smartbug_wil

## 💾 8. Lưu mô hình

In [9]:
os.makedirs(SAVE_DIR, exist_ok=True)
torch.save(model.state_dict(), os.path.join(SAVE_DIR, "model_state.pt"))
tokenizer.save_pretrained(SAVE_DIR)
json.dump({"model_name": MODEL_NAME, "tokenizer_name": TOKENIZER_NAME, "max_len": MAX_LEN,
           "max_chunks": MAX_CHUNKS, "hidden": HIDDEN, "use_lora": USE_LORA,
           "cats": CATS, "threshold": best_t, "label2id": LABEL2ID},
          open(os.path.join(SAVE_DIR, "infer_config.json"), "w"), indent=2)
print("Đã lưu:", os.listdir(SAVE_DIR))

Đã lưu: ['tokenizer.json', 'special_tokens_map.json', 'model_state.pt', 'merges.txt', 'infer_config.json', 'vocab.json', 'tokenizer_config.json']


## 🔍 9. Inference thử

In [10]:
model.eval()
@torch.no_grad()
def predict_contract(code, threshold=None):
    threshold = best_t if threshold is None else threshold
    iid, am = chunk_encode(code)
    out = model(input_ids=iid.to(device), attention_mask=am.to(device),
                n_chunks=torch.tensor([iid.shape[0]]).to(device))
    p = torch.softmax(out.logits, dim=-1)[0, 1].item()
    return ("Vulnerable" if p >= threshold else "Safe"), p

TEST_CODE = """
function withdraw(uint amount) public {
    require(balances[msg.sender] >= amount);
    (bool success,) = msg.sender.call{value: amount}("");
    require(success);
    balances[msg.sender] -= amount;   // giảm số dư SAU khi gửi -> reentrancy
}
"""
lbl, prob = predict_contract(TEST_CODE)
print(f"Dự đoán: {lbl}  (P(Vulnerable)={prob:.3f}, ngưỡng={best_t:.3f})")

Dự đoán: Safe  (P(Vulnerable)=0.071, ngưỡng=0.160)
